# j-lens

Minimal implementation of the Jacobian lens from
[*Verbalizable Representations Form a Global Workspace in Language Models*](https://transformer-circuits.pub/2026/workspace/index.html) (Gurnee et al., 2026), on Qwen3.5-4B.

The lens transports a residual-stream activation $h_\ell$ into the final-layer basis with the corpus-averaged Jacobian $J_\ell = \mathbb{E}\left[\partial h_{\text{final}} / \partial h_\ell\right]$ and decodes it with the model's own unembedding: $\text{lens}(h_\ell) = \mathrm{softmax}(W_U\, \mathrm{norm}(J_\ell h_\ell))$.

In [ ]:
import os
import torch

import evals
import jlens
import interventions as iv

MODEL = "Qwen/Qwen3.5-4B"  # or "Qwen/Qwen3.5-4B-Base"
LENS_PATH = "lens.pt"
N_FIT_PROMPTS = 100  # paper: 10 already beats baselines, released lenses use 1000

## 1. Model and lens

Fit $J_\ell$ on WikiText-103, or load a previous fit. A pre-fitted lens for this model is also on the Hub (`neuronpedia/jacobian-lens`, revision `qwen-n1000`).

In [ ]:
model, tok = jlens.load_model(MODEL)
n = len(model.model.layers)

if os.path.exists(LENS_PATH):
    lens = jlens.JLens.load(LENS_PATH)
else:
    lens = jlens.fit(model, tok, jlens.load_wikitext(N_FIT_PROMPTS),
                     dim_batch=8, checkpoint="fit_ckpt.pt")
    lens.save(LENS_PATH)
lens

## 2. Readout: J-lens vs logit lens

Prompts with an unspoken intermediate, read at the last token across layers. The J-lens surfaces the intermediate (*Italy*'s currency; the English *small* behind a Spanish answer) at depths where the logit lens is still noise (§2.4, §A.5).

In [ ]:
def show(prompt, layers):
    jl, _, _ = lens.readout(model, tok, prompt, layers=layers, positions=[-1])
    ll, ml, _ = lens.readout(model, tok, prompt, layers=layers, positions=[-1], use_jacobian=False)
    top = lambda lg: [tok.decode([t]) for t in lg.topk(5).indices]
    for l in reversed(layers):
        print(f"L{l:>2}  J-lens: {top(jl[l][0])!r:<62} logit-lens: {top(ll[l][0])!r}")
    print("model output:", top(ml[0]))

layers = sorted({round(x * (n - 2)) for x in (0.2, 0.35, 0.5, 0.65, 0.8, 0.95)})
show("Fact: The currency used in the country shaped like a boot is the", layers)

In [ ]:
show('Lo opuesto de "grande" es "', layers)
# the unspoken intermediate is the *English* " small" (L24-28) —
# the model computes in English and answers in Spanish (paper §3.3, Fig 12)

## 3. Workspace band

Per-layer signatures from §4.1 over held-out WikiText: excess kurtosis of the readout, top-10 agreement with the model's next token, and top-1 autocorrelation vs a shuffled null. The workspace is the band where kurtosis and autocorrelation are high but the readout has not yet collapsed onto the next token ("motor" layers).

In [ ]:
import matplotlib.pyplot as plt

m = evals.band_metrics(model, tok, lens, jlens.load_wikitext(120)[100:])
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
for ax, key in zip(axes, ["kurtosis", "next_token_agree", "autocorr"]):
    ax.plot(list(m), [v[key] for v in m.values()])
    ax.set_title(key); ax.set_xlabel("layer")
plt.tight_layout()

WORKSPACE = list(range(round(0.38 * n), round(0.92 * n)))  # adjust from the curves

## 4. pass@k evaluation

The six eval sets of §A.6. An intermediate is recovered at $k$ if its min-over-layers lens rank is $\le k$; the summary is the AUC of pass@$k$ against $\log k$, normalized so always-rank-1 scores 1.

In [ ]:
for name in evals.EVALS:
    _, auc_j = evals.pass_at_k(model, tok, lens, name)
    _, auc_l = evals.pass_at_k(model, tok, lens, name, use_jacobian=False)
    print(f"{name:<14} J-lens {auc_j:.3f}   logit-lens {auc_l:.3f}")

## 5. Interventions

**Verbal report** (§3.1): the model thinks of a category item; swapping its lens coordinates for another item's across the workspace band should change what it reports.

**Internal reasoning** (§3.3): two-hop prompts; swapping the unspoken intermediate (*Brazil* → *Mexico*) should redirect the answer (*Portuguese* → *Spanish*). Paper numbers at Claude scale: 54–70%; expect less at 4B — swap success tracks model size and workspace loading (§3.4).

In [ ]:
data = evals.fetch("experiments", "verbal-report")["candidates"]
chat = lambda cat: tok.apply_chat_template(
    [{"role": "user", "content": f"Think of a {cat}. Answer in one word."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)

n_ok = n_tot = 0
for cat, words in data.items():
    prompt = chat(cat)
    answer = tok.decode([iv.logits_with(model, tok, prompt).argmax()]).strip()
    for cand in [w for w in words if w.lower() != answer.lower()][:5]:
        pairs = evals.pair_token_ids(tok, answer, cand)
        cand_ids = evals.token_ids_of(tok, cand)
        if not (pairs and cand_ids):
            continue
        n_tot += 1
        swapped = iv.logits_with(model, tok, prompt, iv.swap_edits(lens, model, WORKSPACE, pairs))
        n_ok += swapped.argmax().item() in cand_ids
print(f"verbal report follows the swap on {n_ok}/{n_tot} trials")

In [ ]:
items = evals.fetch("experiments", "probe-swap")["items"]
n_base = n_swap = used = 0
for item in items:
    pairs = evals.pair_token_ids(tok, item["intermediate"], item["swap_to"])
    ans = evals.token_ids_of(tok, item["answer"])
    swap_ans = evals.token_ids_of(tok, item["swap_answer"])
    if not (pairs and ans and swap_ans):
        continue
    used += 1
    prompt = item["prompt"].rstrip()
    n_base += iv.logits_with(model, tok, prompt).argmax().item() in ans
    edits = iv.swap_edits(lens, model, WORKSPACE, pairs)
    n_swap += iv.logits_with(model, tok, prompt, edits).argmax().item() in swap_ans
print(f"baseline correct {n_base}/{used}   swap redirects the answer {n_swap}/{used}")

## 6. J-space decomposition

Sparse nonnegative pursuit of an activation against the layer's lens vectors (§2.3): the atoms name what the model is holding at that position; the remainder is the non-J-space component.

In [ ]:
prompt = "Fact: The number of legs on the animal that spins webs is"
layer = WORKSPACE[len(WORKSPACE) // 2]
with jlens.record_residuals(model, [layer]) as rec, torch.no_grad():
    model.model(input_ids=jlens.encode(model, tok, prompt), use_cache=False)
h = rec.acts[layer][0, -1]

ids, coefs, recon = lens.decompose(model, layer, h, k=16)
print([(tok.decode([t]), round(c.item(), 2)) for t, c in zip(ids, coefs)])
print(f"J-space component: {(recon.norm() / h.float().cpu().norm()).item():.1%} of activation norm")